In [1]:
import pandas as pd
import numpy as np
import pickle
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import os

tqdm.pandas()
os.makedirs('data/processed', exist_ok=True)


In [2]:
print("Loading MovieLens files...")
ml_path = 'data/ml-25m/'
ratings = pd.read_csv(ml_path + 'ratings.csv')
movies = pd.read_csv(ml_path + 'movies.csv')
links = pd.read_csv(ml_path + 'links.csv')

print("Loading TMDB files...")
tmdb_path = 'data/tmdb/'
tmdb_movies = pd.read_csv(tmdb_path + 'tmdb_5000_movies.csv')


Loading MovieLens files...
Loading TMDB files...


In [3]:
print(f"Original ratings count: {len(ratings)}")

# 1. Keep movies with >= 100 ratings
movie_counts = ratings['movieId'].value_counts()
popular_movies = movie_counts[movie_counts >= 100].index
ratings = ratings[ratings['movieId'].isin(popular_movies)]

# 2. Keep users with >= 50 ratings
user_counts = ratings['userId'].value_counts()
active_users = user_counts[user_counts >= 50].index
ratings = ratings[ratings['userId'].isin(active_users)]

# 3. Create implicit feedback (rating >= 4.0 is considered a "like")
ratings['liked'] = (ratings['rating'] >= 4.0).astype(int)
positive_ratings = ratings[ratings['liked'] == 1][['userId', 'movieId', 'timestamp']]

print(f"Filtered positive ratings: {len(positive_ratings)}")
print(f"Unique Users: {positive_ratings['userId'].nunique()}, Unique Movies: {positive_ratings['movieId'].nunique()}")


Original ratings count: 25000095
Filtered positive ratings: 11158595
Unique Users: 102128, Unique Movies: 10326


In [4]:
# Create contiguous IDs mapping
user_ids = positive_ratings['userId'].unique()
movie_ids = positive_ratings['movieId'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
movie2idx = {m: i for i, m in enumerate(movie_ids)}

positive_ratings['user_idx'] = positive_ratings['userId'].map(user2idx)
positive_ratings['movie_idx'] = positive_ratings['movieId'].map(movie2idx)

# Filter movies dataframe to only include movies that survived filtering
movies_filtered = movies[movies['movieId'].isin(movie_ids)].copy()
movies_filtered['movie_idx'] = movies_filtered['movieId'].map(movie2idx)


In [5]:
# Map TMDB IDs to MovieLens movies
movies_merged = pd.merge(movies_filtered, links[['movieId', 'tmdbId']], on='movieId', how='left')
movies_merged = movies_merged.dropna(subset=['tmdbId'])
movies_merged['tmdbId'] = movies_merged['tmdbId'].astype(int)

# Merge with TMDB text data
tmdb_movies = tmdb_movies.rename(columns={'id': 'tmdbId'})
movies_with_plots = pd.merge(movies_merged, tmdb_movies[['tmdbId', 'overview', 'tagline']], on='tmdbId', how='left')

# Combine Title, Tagline, and Overview into one rich text string
movies_with_plots['overview'] = movies_with_plots['overview'].fillna('')
movies_with_plots['tagline'] = movies_with_plots['tagline'].fillna('')
movies_with_plots['text_feature'] = movies_with_plots['title'] + ". " + \
                                    movies_with_plots['tagline'] + " " + \
                                    movies_with_plots['overview']

print(f"Movies ready for embedding: {len(movies_with_plots)}")
# Get the valid movie_idxs that survived the TMDB merge
valid_movie_idxs = set(movies_with_plots['movie_idx'].unique())

# Filter positive_ratings to REMOVE interactions with the 5 missing movies
positive_ratings = positive_ratings[positive_ratings['movie_idx'].isin(valid_movie_idxs)]
print(f"Cleaned interactions: {len(positive_ratings)}")


Movies ready for embedding: 10321
Cleaned interactions: 11158008


In [6]:
print("Loading all-MiniLM-L6-v2...")
# Using a lightweight, fast, and highly effective model
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating text embeddings (this may take a few minutes)...")
texts = movies_with_plots['text_feature'].tolist()
embeddings = model.encode(texts, show_progress_bar=True, batch_size=256)

# Create a dictionary mapping the safe 'movie_idx' to its embedding vector
movie_idx_to_embedding = {
    row['movie_idx']: embeddings[i] 
    for i, row in movies_with_plots.iterrows()
}


Loading all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating text embeddings (this may take a few minutes)...


Batches:   0%|          | 0/41 [00:00<?, ?it/s]

In [7]:
print("Saving prepared datasets to data/processed/...")

# 1. Save interactions
positive_ratings = positive_ratings.sort_values(by=['user_idx', 'timestamp'])
from sklearn.model_selection import train_test_split

train_interactions, val_interactions = train_test_split(
    positive_ratings[['user_idx', 'movie_idx', 'timestamp']], 
    test_size=0.2, 
    random_state=42
)

train_interactions.to_csv('data/processed/train_interactions.csv', index=False)
val_interactions.to_csv('data/processed/val_interactions.csv', index=False)

# 2. Save ID Mappings (Crucial for later identifying recommendations)
with open('data/processed/user2idx.pkl', 'wb') as f:
    pickle.dump(user2idx, f)
with open('data/processed/movie2idx.pkl', 'wb') as f:
    pickle.dump(movie2idx, f)


# 3. Save Embeddings
with open('data/processed/movie_embeddings.pkl', 'wb') as f:
    pickle.dump(movie_idx_to_embedding, f)
    
# 4. Save metadata for inference
movies_with_plots[['movie_idx', 'movieId', 'title', 'genres']].to_csv('data/processed/movie_metadata.csv', index=False)

print("Done")

Saving prepared datasets to data/processed/...
Done
